In [3]:
#!pip install optuna

In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [7]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2024-11-05 08:36:51,133] A new study created in memory with name: no-name-4953ff66-99aa-40b4-938e-6eb0c512a6fc
[I 2024-11-05 08:36:51,811] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 190, 'max_depth': 8}. Best is trial 0 with value: 0.7616387337057727.
[I 2024-11-05 08:36:52,155] Trial 1 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 102, 'max_depth': 3}. Best is trial 0 with value: 0.7616387337057727.
[I 2024-11-05 08:36:52,386] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 51, 'max_depth': 16}. Best is trial 2 with value: 0.7709497206703911.
[I 2024-11-05 08:36:52,929] Trial 3 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 128, 'max_depth': 20}. Best is trial 3 with value: 0.7765363128491621.
[I 2024-11-05 08:36:53,228] Trial 4 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 79, 'max_depth': 11}. Best is trial 3 with value: 0.77653631

[I 2024-11-05 08:37:11,410] Trial 45 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 111, 'max_depth': 19}. Best is trial 9 with value: 0.7783985102420857.
[I 2024-11-05 08:37:11,759] Trial 46 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 94, 'max_depth': 16}. Best is trial 9 with value: 0.7783985102420857.
[I 2024-11-05 08:37:12,332] Trial 47 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 162, 'max_depth': 14}. Best is trial 9 with value: 0.7783985102420857.
[I 2024-11-05 08:37:12,851] Trial 48 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 143, 'max_depth': 18}. Best is trial 9 with value: 0.7783985102420857.
[I 2024-11-05 08:37:13,266] Trial 49 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 128, 'max_depth': 3}. Best is trial 9 with value: 0.7783985102420857.


In [8]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 122, 'max_depth': 15}


In [10]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(n_estimators=122,max_depth=15, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


## Samplers in Optuna <br>
#### 1)TPE <br>
#### 2)RandomSampler <br>
#### 3)GrideSearch

In [ ]:
#RandomSampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [11]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2024-11-05 08:53:52,434] A new study created in memory with name: no-name-cf873eb3-c44b-4391-846d-5dcbb2da7b05
[I 2024-11-05 08:53:52,868] Trial 0 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 141, 'max_depth': 3}. Best is trial 0 with value: 0.7579143389199254.
[I 2024-11-05 08:53:53,229] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 107, 'max_depth': 10}. Best is trial 1 with value: 0.7635009310986964.
[I 2024-11-05 08:53:53,717] Trial 2 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 148, 'max_depth': 6}. Best is trial 1 with value: 0.7635009310986964.
[I 2024-11-05 08:53:54,209] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 148, 'max_depth': 9}. Best is trial 1 with value: 0.7635009310986964.
[I 2024-11-05 08:53:54,424] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 59, 'max_depth': 12}. Best is trial 1 with value: 0.76350093

[I 2024-11-05 08:54:11,841] Trial 45 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 131, 'max_depth': 18}. Best is trial 29 with value: 0.7783985102420857.
[I 2024-11-05 08:54:12,218] Trial 46 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 106, 'max_depth': 9}. Best is trial 29 with value: 0.7783985102420857.
[I 2024-11-05 08:54:12,441] Trial 47 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 56, 'max_depth': 18}. Best is trial 29 with value: 0.7783985102420857.
[I 2024-11-05 08:54:12,855] Trial 48 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 117, 'max_depth': 11}. Best is trial 29 with value: 0.7783985102420857.
[I 2024-11-05 08:54:13,358] Trial 49 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 149, 'max_depth': 10}. Best is trial 29 with value: 0.7783985102420857.


In [12]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 125, 'max_depth': 14}


In [14]:
# GrideSearch
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [15]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2024-11-05 08:55:15,374] A new study created in memory with name: no-name-dd5b0fe3-723f-4936-9ddc-ebfcee070739
[I 2024-11-05 08:55:15,706] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2024-11-05 08:55:16,227] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2024-11-05 08:55:16,445] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2024-11-05 08:55:16,803] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2024-11-05 08:55:17,159] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [16]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


https://chatgpt.com/share/67299115-c45c-8000-af1e-5be53e59f665

## Optuna Visualizations

In [22]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2024-11-05 09:15:02,945] A new study created in memory with name: no-name-5c85653b-200c-49f4-adc9-a7062da36d74
[I 2024-11-05 09:15:03,226] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 74, 'max_depth': 3}. Best is trial 0 with value: 0.7616387337057727.
[I 2024-11-05 09:15:03,914] Trial 1 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 196, 'max_depth': 9}. Best is trial 0 with value: 0.7616387337057727.
[I 2024-11-05 09:15:04,431] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 148, 'max_depth': 13}. Best is trial 2 with value: 0.7709497206703911.
[I 2024-11-05 09:15:04,685] Trial 3 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 74, 'max_depth': 8}. Best is trial 3 with value: 0.7728119180633147.
[I 2024-11-05 09:15:05,092] Trial 4 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 116, 'max_depth': 20}. Best is trial 4 with value: 0.778398510

[I 2024-11-05 09:15:23,414] Trial 45 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 144, 'max_depth': 12}. Best is trial 5 with value: 0.7802607076350093.
[I 2024-11-05 09:15:24,076] Trial 46 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 195, 'max_depth': 14}. Best is trial 5 with value: 0.7802607076350093.
[I 2024-11-05 09:15:24,283] Trial 47 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 57, 'max_depth': 16}. Best is trial 5 with value: 0.7802607076350093.
[I 2024-11-05 09:15:24,876] Trial 48 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 174, 'max_depth': 19}. Best is trial 5 with value: 0.7802607076350093.
[I 2024-11-05 09:15:25,234] Trial 49 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 101, 'max_depth': 17}. Best is trial 5 with value: 0.7802607076350093.


In [23]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 196, 'max_depth': 16}


In [24]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [25]:
# 1. Optimization History
plot_optimization_history(study).show()

In [26]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [27]:
# 3. Slice Plot
plot_slice(study).show()

In [28]:
# 4. Contour Plot
plot_contour(study).show()

In [29]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

### Optimizing Multiple ML Models

In [30]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [31]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [32]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=100)  # Run 100 trials to find the best hyperparameters

[I 2024-11-05 11:19:59,660] A new study created in memory with name: no-name-1d1dc9cd-debb-41ad-9539-305a1cd5feee
[I 2024-11-05 11:20:00,208] Trial 0 finished with value: 0.7728119180633147 and parameters: {'classifier': 'RandomForest', 'n_estimators': 201, 'max_depth': 13, 'min_samples_split': 8, 'min_samples_leaf': 8, 'bootstrap': False}. Best is trial 0 with value: 0.7728119180633147.
[I 2024-11-05 11:20:01,264] Trial 1 finished with value: 0.6983240223463687 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 110, 'learning_rate': 0.022961874974007902, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.7728119180633147.
[I 2024-11-05 11:20:01,464] Trial 2 finished with value: 0.7430167597765364 and parameters: {'classifier': 'RandomForest', 'n_estimators': 79, 'max_depth': 3, 'min_samples_split': 8, 'min_samples_leaf': 3, 'bootstrap': False}. Best is trial 0 with value: 0.7728119180633147.
[I 2024-11-05 11:20:02,435] Trial 

[I 2024-11-05 11:20:09,237] Trial 33 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 85.59077942109079, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:09,300] Trial 34 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 13.898100092907576, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:10,644] Trial 35 finished with value: 0.7281191806331471 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 237, 'learning_rate': 0.18843031680174518, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 7}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:10,671] Trial 36 finished with value: 0.7374301675977654 and parameters: {'classifier': 'SVM', 'C': 0.3803426688139391, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:11

[I 2024-11-05 11:20:19,781] Trial 68 finished with value: 0.7467411545623835 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 213, 'learning_rate': 0.022971803439813074, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 4}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:19,875] Trial 69 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 32.88417291870171, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:19,917] Trial 70 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 6.534768683170833, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:20,171] Trial 71 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 98.67466599010231, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 7 with value: 0.7858472998137801.
[I 2024-11-05 11:20:20,

In [33]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.2520324970435341, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7858472998137803


In [34]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.772812,2024-11-05 11:19:59.661361,2024-11-05 11:20:00.208606,0 days 00:00:00.547245,NaN,False,RandomForest,NaN,NaN,NaN,13.0,8.0,8.0,201.0,COMPLETE
1,1,0.698324,2024-11-05 11:20:00.209606,2024-11-05 11:20:01.263471,0 days 00:00:01.053865,NaN,NaN,GradientBoosting,NaN,NaN,0.022962,10.0,1.0,4.0,110.0,COMPLETE
2,2,0.743017,2024-11-05 11:20:01.265750,2024-11-05 11:20:01.463356,0 days 00:00:00.197606,NaN,False,RandomForest,NaN,NaN,NaN,3.0,3.0,8.0,79.0,COMPLETE
3,3,0.767225,2024-11-05 11:20:01.465356,2024-11-05 11:20:02.435975,0 days 00:00:00.970619,NaN,NaN,GradientBoosting,NaN,NaN,0.010665,17.0,10.0,3.0,221.0,COMPLETE
4,4,0.718808,2024-11-05 11:20:02.436979,2024-11-05 11:20:02.469421,0 days 00:00:00.032442,4.826474,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2024-11-05 11:20:23.824605,2024-11-05 11:20:23.850583,0 days 00:00:00.025978,0.252032,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.782123,2024-11-05 11:20:23.850583,2024-11-05 11:20:23.882911,0 days 00:00:00.032328,0.366926,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.744879,2024-11-05 11:20:23.883930,2024-11-05 11:20:25.091341,0 days 00:00:01.207411,NaN,NaN,GradientBoosting,NaN,NaN,0.025382,10.0,4.0,9.0,170.0,COMPLETE
98,98,0.785847,2024-11-05 11:20:25.093343,2024-11-05 11:20:25.168329,0 days 00:00:00.074986,19.644885,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [35]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 77
GradientBoosting    12
RandomForest        11
Name: count, dtype: int64

In [36]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.744258
RandomForest        0.762147
SVM                 0.768894
Name: value, dtype: float64

In [37]:
# 1. Optimization History
plot_optimization_history(study).show()

In [38]:
# 3. Slice Plot
plot_slice(study).show()

In [39]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()